In [ ]:
!pip install -q "huggingface_hub==0.23.4" "transformers==4.44.0" "sentence-transformers==3.0.1"
!pip install -q "bitsandbytes>=0.46.1" "peft" "accelerate"
!pip install -q "langchain-huggingface" "langchain-chroma" "langchain-text-splitters" "langchain-community" "chromadb"
!pip install -q "pypdf"

In [ ]:
import os, shutil, warnings, torch
warnings.filterwarnings("ignore")

from huggingface_hub import login
os.environ["HF_TOKEN"] = "your_key_here"
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=False,
    repetition_penalty=1.15,
    pad_token_id=tokenizer.eos_token_id,
)

print(f"Model ready — GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

2026-06-24 14:43:20.257213: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782312200.630570     149 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782312200.759748     149 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782312201.759431     149 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782312201.759477     149 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782312201.759480     149 computation_placer.cc:177] computation placer alr

Loading base model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Model ready — GPU: 0.96 GB


In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

PDF_DIR = "/kaggle/input/datasets/teslaincarnate/scd-pdfs"
NEW_VECTORDB_PATH = "/kaggle/working/scd_guidelines_v2"

# Find PDFs
pdf_files = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pdf"):
            pdf_files.append(os.path.join(root, f))
print(f"Found {len(pdf_files)} PDFs")

# Load
print("Loading PDFs...")
all_docs = []
for pdf_path in pdf_files:
    try:
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        for doc in docs:
            doc.metadata["source"] = os.path.basename(pdf_path)
        all_docs.extend(docs)
        print(f"  ✓ {os.path.basename(pdf_path)} — {len(docs)} pages")
    except Exception as e:
        print(f"  ✗ {os.path.basename(pdf_path)} — {e}")

print(f"\nTotal pages: {len(all_docs)}")

# Chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents(all_docs)
print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks)/len(chunks):.0f} chars")

# Embed
print("\nLoading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Build vectordb
if os.path.exists(NEW_VECTORDB_PATH):
    shutil.rmtree(NEW_VECTORDB_PATH)

print("Building vectordb (5-10 mins)...")
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=NEW_VECTORDB_PATH
)
print(f"Done — {vectordb._collection.count()} chunks indexed")

Found 8 PDFs
Loading PDFs...
  ✓ nihms-1562670.pdf — 39 pages
  ✓ IJNS-05-00020.pdf — 15 pages
  ✓ SCD-Guideline-Final-2nd-Edition.pdf — 148 pages
  ✓ ANEMIA2015-791498.pdf — 21 pages
  ✓ Standards-for-the-Clinical-Care-of-Adults-with-Sickle-Cell-in-the-UK-2018.pdf — 310 pages
  ✓ main.pdf — 9 pages
  ✓ sickle-cell-disease-report 020816_0.pdf — 161 pages
  ✓ SPARCo_SoC_SCD_GuidelinesR_Aug2023.pdf — 49 pages

Total pages: 752
Total chunks: 2764
Avg chunk size: 759 chars

Loading embedding model...
Building vectordb (5-10 mins)...
Done — 2764 chunks indexed


In [6]:
SYSTEM = (
    "You are a medical AI assistant specialised in sickle cell disease. "
    "Answer using ONLY the guideline text provided. "
    "Cite sources by document name only — never generate URLs or links. "
    "Be concise and stop after answering the question. "
    "Do not add disclaimers or caveats."
)

def answer_with_rag(question, case=""):
    search_query = f"{case} {question}" if case else question
    retrieved = vectordb.similarity_search(search_query, k=3)

    context = "\n\n".join([
        f"[{doc.metadata.get('source', 'guideline')}]\n{doc.page_content[:600]}"
        for doc in retrieved
    ])

    clinical_content = (
        f"Clinical case: {case}\nQuestion: {question}"
        if case else f"Question: {question}"
    )

    user_content = f"""Guidelines:
{context}

{clinical_content}

Answer in 3-5 bullet points. After each point write (Source: document name). Do not include any URLs."""

    prompt = f"<|user|>\n{user_content}<|end|>\n<|assistant|>\n"

    output = pipe(prompt, max_new_tokens=300, do_sample=False, repetition_penalty=1.15)
    response = output[0]["generated_text"].split("<|assistant|>")[-1].strip()
    response = response.replace("<|end|>", "").strip()
    sources = [doc.metadata.get("source", "unknown") for doc in retrieved]
    return response, sources

In [7]:
test_questions = [
    {"question": "What is the hydroxyurea monitoring protocol for adults with sickle cell disease?", "case": ""},
    {"question": "What is the diagnosis and immediate management in priority order?", "case": "A 4-year-old boy with HbSS presents with fever of 39.2°C and swollen tender hands and feet bilaterally."},
    {"question": "What are the indications for exchange transfusion in sickle cell disease?", "case": ""},
    {"question": "What monitoring intervals are recommended for adults stabilising on hydroxyurea?", "case": ""},
    {"question": "What newborn screening is recommended for sickle cell disease in Nigeria?", "case": ""},
]

for i, q in enumerate(test_questions):
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {q['question']}")
    if q['case']:
        print(f"Case: {q['case']}")
    response, sources = answer_with_rag(q['question'], q['case'])
    print(f"Sources: {sources}")
    print(f"\nResponse:\n{response}")


Q1: What is the hydroxyurea monitoring protocol for adults with sickle cell disease?


You are not running the flash-attention implementation, expect numerical differences.


Sources: ['Standards-for-the-Clinical-Care-of-Adults-with-Sickle-Cell-in-the-UK-2018.pdf', 'sickle-cell-disease-report 020816_0.pdf', 'Standards-for-the-Clinical-Care-of-Adults-with-Sickle-Cell-in-the-UK-2018.pdf']

Response:
- Establish individualized starting dose based on patient's age, weight, hemoglobin level, baseline fetal hemoglobin percentage, renal function, liver enzyme levels, blood pressure control status, history of prior treatment complications such as bone marrow suppression/myelosuppression syndrome; adjust according to tolerance and therapeutic effectiveness over time. (Standard Protocol Monitoring Source Document Name)
   
   *Note: The specific source names are placeholders since actual URL links were excluded per instructions.*

- Regularly monitor complete blood count including reticulocyte counts at least every three months initially after commencing therapy then less frequently once stable regimen identified if no adverse effects occur during initial phase. Asse